# Banking Transaction Intelligence Model
## Fraud Risk & P&L Analytics — Live Demo

**Production-grade fraud detection system.** This notebook walks through:
1. Real-time transaction scoring (19-rule engine + ML ensemble, <100ms)
2. Dataset-level fraud analytics (50,000 transactions)
3. P&L impact breakdown
4. Live REST API calls

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linewidth':   0.6,
    'font.family':      'monospace',
})

TIER_COLORS = {
    'CRITICAL':  '#ff4444',
    'VERY HIGH': '#ff8800',
    'HIGH':      '#ffcc00',
    'MEDIUM':    '#4488ff',
    'LOW':       '#00cc66',
}
print('Libraries loaded.')

## 1. Load Trained ML Models

In [ ]:
from bti.scoring.model_loader import load_models
from bti.scoring.realtime import score_transaction, RULE_WEIGHTS

bundle = load_models()

print(f'Models loaded from: {bundle.manifest_path}')
print(f'Trained at        : {bundle.trained_at}')
print(f'Feature columns   : {len(bundle.feature_cols)}')
print(f'Label encoders    : {list(bundle.label_encoders.keys())}')
print()
print('Model Performance:')
print(f'  Logistic Regression  ROC-AUC : {bundle.lr_roc_auc:.4f}')
print(f'  Random Forest        ROC-AUC : {bundle.rf_roc_auc:.6f}')
print(f'  Random Forest        F1      : {bundle.rf_f1:.4f}')

## 2. Real-Time Transaction Scoring

Three transactions — high-risk, medium-risk, and normal — scored through the full pipeline.

In [ ]:
transactions = {
    'HIGH-RISK (Crypto, 2am, 28× avg)': {
        'transaction_id': 'DEMO-001',
        'customer_id': 'CUST-DEMO',
        'transaction_amount': 12500.0,
        'transaction_time': '02:47:00',
        'channel': 'API/Open Banking',
        'merchant_category': 'Crypto Exchanges',
        'customer_segment': 'Mass Market',
        'risk_score': 72,
        'historical_average_transaction_amount': 450.0,
        'failed_attempt_count': 4,
        'login_attempts': 6,
        'debit_credit_flag': 'Debit',
        'device_id': 'DEV-X9921-UNKNOWN',
        'ip_location': '203.0.113.45',
    },
    'MEDIUM-RISK (Luxury, late night)': {
        'transaction_id': 'DEMO-002',
        'customer_id': 'CUST-DEMO',
        'transaction_amount': 2800.0,
        'transaction_time': '23:15:00',
        'channel': 'Web',
        'merchant_category': 'Luxury Goods',
        'customer_segment': 'Mass Market',
        'risk_score': 45,
        'historical_average_transaction_amount': 600.0,
        'failed_attempt_count': 1,
        'login_attempts': 2,
        'debit_credit_flag': 'Debit',
        'device_id': 'DEV-LAPTOP-HOME',
        'ip_location': '195.0.1.5',
    },
    'LOW-RISK (Grocery, normal hours)': {
        'transaction_id': 'DEMO-003',
        'customer_id': 'CUST-SAFE',
        'transaction_amount': 55.0,
        'transaction_time': '14:30:00',
        'channel': 'Mobile App',
        'merchant_category': 'Grocery & Supermarkets',
        'customer_segment': 'Mass Market',
        'risk_score': 12,
        'historical_average_transaction_amount': 60.0,
        'failed_attempt_count': 0,
        'login_attempts': 1,
        'debit_credit_flag': 'Debit',
        'device_id': 'DEV-IPHONE14-HOME',
        'ip_location': '192.168.1.1',
    },
}

results = {}
for label, txn in transactions.items():
    r = score_transaction(txn, bundle)
    results[label] = r
    color = TIER_COLORS.get(r.final_alert_tier, '#ffffff')
    print(f'\033[1m{label}\033[0m')
    print(f'  Final Score  : {r.final_risk_score:6.2f}/100')
    print(f'  Alert Tier   : {r.final_alert_tier}')
    print(f'  Rules Fired  : {r.rules_triggered} → {r.rules_fired}')
    print(f'  LR Fraud P   : {r.ml_lr_proba:.4f}  |  RF Fraud P: {r.ml_rf_proba:.4f}')
    print(f'  Scored In    : {r.processing_time_ms:.1f}ms')
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Real-Time Fraud Score Breakdown', fontsize=14, fontweight='bold', color='#e6edf3', y=1.02)

for ax, (label, r) in zip(axes, results.items()):
    tier_color = TIER_COLORS.get(r.final_alert_tier, '#ffffff')
    
    # Score gauge as horizontal bar
    ax.barh(['ML Score', 'Rule Score', 'Final Score'],
            [r.ml_anomaly_score, r.fraud_rule_score, r.final_risk_score],
            color=['#4488ff', '#ffcc00', tier_color],
            height=0.5, alpha=0.9)
    ax.set_xlim(0, 100)
    ax.axvline(x=85, color='#ff4444', linestyle='--', alpha=0.5, linewidth=1)
    ax.axvline(x=70, color='#ff8800', linestyle='--', alpha=0.5, linewidth=1)
    ax.axvline(x=50, color='#ffcc00', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_title(f'{label}\n{r.final_alert_tier} ({r.final_risk_score:.1f})',
                 color=tier_color, fontsize=10, fontweight='bold')
    ax.set_xlabel('Score (0–100)')
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate([r.ml_anomaly_score, r.fraud_rule_score, r.final_risk_score]):
        ax.text(min(v + 1, 95), i, f'{v:.1f}', va='center', fontsize=9, color='#e6edf3')

plt.tight_layout()
plt.savefig('../outputs/charts/demo_score_breakdown.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 3. Rule Engine Anatomy

Which rules fired on the high-risk transaction, and what weight did each carry?

In [ ]:
high_risk_result = results['HIGH-RISK (Crypto, 2am, 28× avg)']

fired_weights   = {r: RULE_WEIGHTS[r] for r in high_risk_result.rules_fired if r in RULE_WEIGHTS}
unfired_weights = {r: RULE_WEIGHTS[r] for r in RULE_WEIGHTS if r not in high_risk_result.rules_fired}

fig, ax = plt.subplots(figsize=(14, 7))

all_rules  = list(RULE_WEIGHTS.keys())
all_weights = [RULE_WEIGHTS[r] for r in all_rules]
colors = ['#ff4444' if r in fired_weights else '#2d333b' for r in all_rules]

bars = ax.barh(all_rules, all_weights, color=colors, height=0.65, alpha=0.9)

for bar, rule, weight in zip(bars, all_rules, all_weights):
    label_color = '#e6edf3' if rule in fired_weights else '#484f58'
    fired_label = ' ◀ FIRED' if rule in fired_weights else ''
    ax.text(weight + 0.05, bar.get_y() + bar.get_height()/2,
            f'{weight}{fired_label}', va='center', fontsize=9, color=label_color)

ax.set_xlabel('Rule Weight')
ax.set_title(f'Fraud Rule Engine — HIGH-RISK transaction ({len(fired_weights)}/19 rules fired)',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, 12)
ax.grid(axis='x', alpha=0.3)

fired_patch   = mpatches.Patch(color='#ff4444', label='Fired')
unfired_patch = mpatches.Patch(color='#2d333b', label='Not fired')
ax.legend(handles=[fired_patch, unfired_patch], loc='lower right')

plt.tight_layout()
plt.savefig('../outputs/charts/demo_rule_engine.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'Weighted rule score: {high_risk_result.fraud_rule_score:.2f}/100')

## 4. Dataset Overview — 50,000 Transactions

In [ ]:
df = pd.read_csv('../data/processed/banking_transactions_ml_scored.csv', low_memory=False)
print(f'Dataset shape     : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Date range        : {df["transaction_date"].min()} → {df["transaction_date"].max()}')
print(f'Total volume      : £{df["transaction_amount"].sum():,.0f}')
print(f'Fraud rate        : {df["fraud_flag"].mean()*100:.2f}%  ({int(df["fraud_flag"].sum()):,} transactions)')
print(f'Channels          : {df["channel"].nunique()}')
print(f'Merchant cats     : {df["merchant_category"].nunique()}')
print(f'Customer segments : {df["customer_segment"].nunique()}')
print()
print('Alert tier distribution:')
for tier, count in df['final_alert_tier'].value_counts().items():
    bar = '█' * (count // 500)
    print(f'  {tier:<12} {count:>6,}  {bar}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Banking Transaction Intelligence — Dataset Analytics', fontsize=15, fontweight='bold', y=1.01)

# 1. Alert tier distribution
ax = axes[0, 0]
tier_order = ['CRITICAL', 'VERY HIGH', 'HIGH', 'MEDIUM', 'LOW']
tier_counts = df['final_alert_tier'].value_counts().reindex(tier_order).fillna(0)
colors = [TIER_COLORS[t] for t in tier_order]
ax.bar(tier_order, tier_counts.values, color=colors, alpha=0.9, edgecolor='#30363d')
ax.set_title('Alert Tier Distribution', fontweight='bold')
ax.set_ylabel('Transactions')
for i, v in enumerate(tier_counts.values):
    ax.text(i, v + 100, f'{int(v):,}', ha='center', fontsize=9)
ax.set_ylim(0, tier_counts.max() * 1.12)

# 2. Final risk score distribution
ax = axes[0, 1]
ax.hist(df['final_risk_score'], bins=50, color='#4488ff', alpha=0.8, edgecolor='#30363d')
ax.axvline(85, color='#ff4444', linestyle='--', label='CRITICAL (85)')
ax.axvline(70, color='#ff8800', linestyle='--', label='VERY HIGH (70)')
ax.axvline(50, color='#ffcc00', linestyle='--', label='HIGH (50)')
ax.set_title('Risk Score Distribution', fontweight='bold')
ax.set_xlabel('Final Risk Score (0–100)')
ax.set_ylabel('Count')
ax.legend(fontsize=8)

# 3. Fraud rate by channel
ax = axes[0, 2]
channel_fraud = df.groupby('channel')['fraud_flag'].mean().sort_values(ascending=True) * 100
ax.barh(channel_fraud.index, channel_fraud.values, color='#ff6b6b', alpha=0.85)
ax.set_title('Fraud Rate by Channel (%)', fontweight='bold')
ax.set_xlabel('Fraud Rate (%)')
for i, v in enumerate(channel_fraud.values):
    ax.text(v + 0.05, i, f'{v:.2f}%', va='center', fontsize=8)

# 4. Monthly P&L impact
ax = axes[1, 0]
df['month'] = pd.to_datetime(df['transaction_date']).dt.to_period('M').astype(str)
monthly_pnl = df.groupby('month')['net_pnl_impact'].sum() / 1e6
colors_pnl = ['#00cc66' if v >= 0 else '#ff4444' for v in monthly_pnl.values]
ax.bar(range(len(monthly_pnl)), monthly_pnl.values, color=colors_pnl, alpha=0.85)
ax.set_title('Monthly Net P&L Impact (£M)', fontweight='bold')
ax.set_ylabel('£ Millions')
ax.set_xticks(range(0, len(monthly_pnl), 3))
ax.set_xticklabels(monthly_pnl.index[::3], rotation=45, ha='right', fontsize=8)
ax.axhline(0, color='#e6edf3', linewidth=0.5)

# 5. Fraud loss by merchant category (top 8)
ax = axes[1, 1]
merch_loss = df.groupby('merchant_category')['fraud_loss'].sum().sort_values(ascending=True).tail(8) / 1e6
ax.barh(merch_loss.index, merch_loss.values, color='#ff4444', alpha=0.8)
ax.set_title('Fraud Loss by Merchant Category (£M)', fontweight='bold')
ax.set_xlabel('£ Millions')
for i, v in enumerate(merch_loss.values):
    ax.text(v + 0.02, i, f'£{v:.1f}M', va='center', fontsize=8)

# 6. ML probability vs rule score scatter
ax = axes[1, 2]
sample = df.sample(n=2000, random_state=42)
tier_col = sample['final_alert_tier'].map(TIER_COLORS).fillna('#8b949e')
ax.scatter(sample['fraud_rule_score'], sample['ml_anomaly_score_norm'],
           c=tier_col, alpha=0.4, s=8)
ax.set_title('Rule Score vs ML Score', fontweight='bold')
ax.set_xlabel('Fraud Rule Score (0–100)')
ax.set_ylabel('ML Anomaly Score (0–100)')
patches = [mpatches.Patch(color=c, label=t) for t, c in TIER_COLORS.items()]
ax.legend(handles=patches, fontsize=7, loc='upper left')

plt.tight_layout()
plt.savefig('../outputs/charts/demo_analytics_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 5. P&L KPIs

In [ ]:
kpis = {
    'Total Transactions' : f"{len(df):,}",
    'Total Volume'       : f"£{df['transaction_amount'].sum()/1e6:,.1f}M",
    'Gross Fee Income'   : f"£{(df['fee_income'].sum() + df['interchange_income'].sum())/1e6:,.1f}M",
    'Total Fraud Loss'   : f"£{df['fraud_loss'].sum()/1e6:,.1f}M",
    'Chargeback Loss'    : f"£{df['chargeback_loss'].sum()/1e6:,.1f}M",
    'Net P&L Impact'     : f"£{df['net_pnl_impact'].sum()/1e6:,.1f}M",
    'Fraud Rate'         : f"{df['fraud_flag'].mean()*100:.2f}%",
    'Fraud Loss Rate'    : f"{df['fraud_loss'].sum()/df['transaction_amount'].sum()*100:.2f}%",
    'CRITICAL Alerts'    : f"{(df['final_alert_tier']=='CRITICAL').sum():,}",
    'VERY HIGH Alerts'   : f"{(df['final_alert_tier']=='VERY HIGH').sum():,}",
}

print('=' * 45)
print('  BANKING TRANSACTION INTELLIGENCE — KPIs')
print('=' * 45)
for k, v in kpis.items():
    print(f'  {k:<25} {v:>12}')
print('=' * 45)

## 6. Live API Call

Start the API server first (`python main.py api`), then run this cell.

In [ ]:
import requests, json

BASE_URL = 'http://localhost:8000'

try:
    health = requests.get(f'{BASE_URL}/health', timeout=3).json()
    print('API Status:', health['status'], '| Version:', health['version'], '| DB:', health['database'])
    print()

    payload = {
        'transaction_id': 'NOTEBOOK-LIVE-001',
        'customer_id': 'CUST-DEMO-NB',
        'transaction_amount': 12500.00,
        'transaction_time': '02:47:00',
        'channel': 'API/Open Banking',
        'merchant_category': 'Crypto Exchanges',
        'customer_segment': 'Mass Market',
        'risk_score': 72,
        'historical_average_transaction_amount': 450.0,
        'failed_attempt_count': 4,
        'login_attempts': 6,
        'debit_credit_flag': 'Debit',
        'device_id': 'DEV-X9921-UNKNOWN',
        'ip_location': '203.0.113.45',
    }

    response = requests.post(f'{BASE_URL}/api/v1/score/', json=payload, timeout=5)
    result   = response.json()

    print('LIVE SCORING RESULT:')
    print(json.dumps(result, indent=2))

except requests.exceptions.ConnectionError:
    print('API not running. Start it with: python main.py api')
    print('Then re-run this cell for a live demo.')

## 7. Customer Risk Profiling

In [ ]:
# Top 10 highest-risk customers
customer_risk = (
    df.groupby('customer_id')
      .agg(
          transactions=('transaction_id', 'count'),
          avg_risk_score=('final_risk_score', 'mean'),
          fraud_count=('fraud_flag', 'sum'),
          total_fraud_loss=('fraud_loss', 'sum'),
          critical_count=('final_alert_tier', lambda x: (x == 'CRITICAL').sum()),
          very_high_count=('final_alert_tier', lambda x: (x == 'VERY HIGH').sum()),
      )
      .sort_values('avg_risk_score', ascending=False)
      .head(10)
      .reset_index()
)
customer_risk['total_fraud_loss'] = customer_risk['total_fraud_loss'].apply(lambda x: f'£{x:,.0f}')
customer_risk['avg_risk_score']   = customer_risk['avg_risk_score'].round(2)
customer_risk.index = range(1, 11)
print('Top 10 Highest-Risk Customers:')
print(customer_risk.to_string())

---
## System Summary

| Component | Detail |
|---|---|
| **Scoring latency** | <100ms per transaction |
| **Rules** | 19 weighted rules (R01–R19) |
| **ML models** | Isolation Forest + Logistic Regression + Random Forest |
| **RF ROC-AUC** | 0.9999 |
| **Features** | 28 (numeric + categorical, saved label encoders) |
| **Dataset** | 50,000 transactions, 24 months |
| **API endpoints** | 18 (FastAPI, <100ms p95) |
| **Test coverage** | 70 tests (unit + integration) |
| **Database** | SQLAlchemy ORM, SQLite/PostgreSQL |
| **Deployment** | Docker + docker-compose + GitHub Actions CI |

**GitHub:** https://github.com/SrijanUpadhyay-dotcom/Banking-Transaction-Intelligence-Model-for-Fraud-Risk-P-L-Analytics